# rebound_state_head_target_recipe_v1

Purpose: convert the `momentum_drawdown` rebound representation into a fold-heldout state-head target-position recipe before any PPO handoff.

Context:
- `state_objective_state_conditioned_loss_v1` found BC action fit can memorize rebound rows but fold-heldout rebound accuracy stayed `0.000`.
- `rebound_representation_separability_v1` found `momentum_drawdown` (`mom_5,mom_21,drawdown_63`) is cross-fold separable for rebound state, with holdout AUC min `0.980`.
- This notebook tests whether a state head score can safely route target positions to `rebound_target_position` while leaving other rows on their default non-rebound target.

External references:
- Qlib harness and workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validation/test discipline: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Auxiliary task / representation decomposition basis: https://arxiv.org/abs/2004.11362 and https://arxiv.org/abs/1807.03748
- Class imbalance caution from prior failed BC loss screen: https://arxiv.org/abs/1901.05555

Governance:
- Baseline: `rebound_representation_separability_v1` and `state_objective_state_conditioned_loss_v1`.
- Role: diagnose / prepare a future training recipe.
- Changed layer: model-selection/training objective target routing, not runtime execution.
- Acceptance: state-head target recipe may pass only as a future PPO recipe; it cannot promote a model or actor by itself.


In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src.soxl_quant.app.representation_diagnostics import (
    StateHeadTargetRecipeConfig,
    score_state_head_target_recipes,
)
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'rebound_state_head_target_recipe_v1'
OUT_DIR = Path('runtime/model_selection/card') / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_PATH = Path('runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv')
REPRESENTATION_SELECTION = Path('runtime/model_selection/card/rebound_representation_separability_v1/selection.json')
LOSS_SELECTION = Path('runtime/model_selection/card/state_objective_state_conditioned_loss_v1/selection.json')

with REPRESENTATION_SELECTION.open() as fh:
    representation_selection = json.load(fh)
with LOSS_SELECTION.open() as fh:
    loss_selection = json.load(fh)

targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
targets['fold'] = targets['fold'].astype(str)
targets['is_rebound_cluster'] = targets['is_rebound_cluster'].fillna(False).astype(bool)
targets['is_downtrend_state'] = targets['is_downtrend_state'].fillna(False).astype(bool)
targets['true_target_position'] = [state_objective_target_position_from_row(row) for row in targets.to_dict('records')]
targets['default_target_position'] = np.where(
    targets['is_downtrend_state'],
    pd.to_numeric(targets['risk_target_position'], errors='coerce'),
    pd.to_numeric(targets['teacher_position'], errors='coerce'),
)

base_context = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_targets': str(TARGETS_PATH),
    'rows': int(len(targets)),
    'fold_counts': {str(k): int(v) for k, v in targets['fold'].value_counts().to_dict().items()},
    'rebound_by_fold': {str(k): int(v) for k, v in targets.groupby('fold')['is_rebound_cluster'].sum().to_dict().items()},
    'representation_best_feature_set': representation_selection['best_feature_set'],
    'representation_holdout_auc_min': float(representation_selection['best_holdout_auc_min']),
    'bc_loss_holdout_rebound_min': float(loss_selection['best_holdout_rebound_accuracy_min']),
}
print(json.dumps(base_context, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "bc_loss_holdout_rebound_min": 0.0,
  "created_at": "2026-05-25T13:34:18.440808+00:00",
  "experiment": "rebound_state_head_target_recipe_v1",
  "fold_counts": {
    "wf_2022_train": 505,
    "wf_2023_2024_train": 756
  },
  "rebound_by_fold": {
    "wf_2022_train": 4,
    "wf_2023_2024_train": 3
  },
  "representation_best_feature_set": "momentum_drawdown",
  "representation_holdout_auc_min": 0.9800796812749004,
  "rows": 1261,
  "source_targets": "runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv"
}


In [2]:

feature_sets = {
    'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63'],
    'momentum_soxx_relative': ['mom_5', 'mom_21', 'drawdown_63', 'soxl_soxx_ratio_widen_21', 'soxx_mom_21'],
    'all_candidate_features': [
        'mom_5', 'mom_21', 'drawdown_63', 'rv_21', 'vol_of_vol_21',
        'soxl_soxx_ratio_widen_21', 'soxl_soxx_ratio_z_252', 'soxx_mom_21',
        'base_position', 'upside_position', 'router_position',
    ],
}
thresholds = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
config = StateHeadTargetRecipeConfig(
    min_recall=0.70,
    min_precision=0.05,
    max_false_positive_rate=0.20,
    max_target_mae=0.02,
    max_rebound_mae=0.03,
    min_covered_folds=2,
)

result = score_state_head_target_recipes(
    targets,
    label_col='is_rebound_cluster',
    fold_col='fold',
    true_target_col='true_target_position',
    default_target_col='default_target_position',
    state_target_col='rebound_target_position',
    feature_sets=feature_sets,
    thresholds=thresholds,
    min_positive_train=2,
    min_positive_eval=2,
    config=config,
)
recipe_scores = result.recipe_scores
row_scores = result.row_scores

recipe_scores.to_csv(OUT_DIR / 'recipe_scores.csv', index=False)
row_scores.to_csv(OUT_DIR / 'row_scores.csv', index=False)

print('recipe scores')
display(recipe_scores)
print('best recipe rows')
display(row_scores[(row_scores['feature_set'] == recipe_scores.iloc[0]['feature_set']) & (row_scores['threshold'] == recipe_scores.iloc[0]['threshold'])].head(20))


recipe scores


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,state_precision,state_recall,true_positive_rate,false_positive_rate,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss
0,momentum_drawdown,0.5,1261,7,1254,2,7,123,0,1131,0.053846,1.000000,1.000000,0.098086,0.000301,0.028571,0.000143,True,0.000000
1,momentum_drawdown,1.5,1261,7,1254,2,5,37,2,1217,0.119048,0.714286,0.714286,0.029506,0.000119,0.021429,0.000000,True,0.000000
2,momentum_soxx_relative,0.5,1261,7,1254,2,7,156,0,1098,0.042945,1.000000,1.000000,0.124402,0.000586,0.028571,0.000430,False,0.014110
3,momentum_soxx_relative,2.0,1261,7,1254,2,6,48,1,1206,0.111111,0.857143,0.857143,0.038278,0.000420,0.050000,0.000143,False,0.020000
4,momentum_drawdown,1.0,1261,7,1254,2,6,70,1,1184,0.078947,0.857143,0.857143,0.055821,0.000420,0.050000,0.000143,False,0.020000
5,momentum_soxx_relative,1.5,1261,7,1254,2,6,81,1,1173,0.068966,0.857143,0.857143,0.064593,0.000420,0.050000,0.000143,False,0.020000
6,momentum_soxx_relative,1.0,1261,7,1254,2,6,111,1,1143,0.051282,0.857143,0.857143,0.088517,0.000563,0.050000,0.000287,False,0.020000
7,momentum_drawdown,0.0,1261,7,1254,2,7,192,0,1062,0.035176,1.000000,1.000000,0.153110,0.000586,0.028571,0.000430,False,0.029648
8,momentum_soxx_relative,0.0,1261,7,1254,2,7,207,0,1047,0.032710,1.000000,1.000000,0.165072,0.001181,0.028571,0.001028,False,0.034579
9,all_candidate_features,-0.5,1261,7,1254,2,6,225,1,1029,0.025974,0.857143,0.857143,0.179426,0.006358,0.050000,0.006114,False,0.068052


best recipe rows


,row_index,feature_set,eval_fold,state_score,coverage_gate,train_positive,eval_positive,eval_negative,is_rebound_cluster,fold,true_target_position,default_target_position,rebound_target_position,threshold,predicted_state,proposed_target,target_error
12610,0,momentum_drawdown,wf_2022_train,-1.042411,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12611,1,momentum_drawdown,wf_2022_train,-1.267392,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12612,2,momentum_drawdown,wf_2022_train,-1.403587,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12613,3,momentum_drawdown,wf_2022_train,-1.192799,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12614,4,momentum_drawdown,wf_2022_train,-1.183788,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12615,5,momentum_drawdown,wf_2022_train,-1.184188,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12616,6,momentum_drawdown,wf_2022_train,-1.123212,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12617,7,momentum_drawdown,wf_2022_train,-0.563825,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12618,8,momentum_drawdown,wf_2022_train,-0.830445,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0
12619,9,momentum_drawdown,wf_2022_train,-1.170011,True,3,4,501,False,wf_2022_train,0.0625,0.0625,0.0625,0.5,False,0.0625,0.0


In [3]:

best = recipe_scores.iloc[0].to_dict() if not recipe_scores.empty else {}
recipe_gate = bool(best.get('recipe_gate', False))
precision = float(best.get('state_precision', np.nan))
recall = float(best.get('state_recall', np.nan))
fpr = float(best.get('false_positive_rate', np.nan))

if recipe_gate:
    decision = 'state_head_target_recipe_passes_pre_ppo_gate_no_runtime_promotion'
    next_mechanism = 'fold_local_ppo_with_aux_rebound_state_head_momentum_drawdown'
    blocker = 'State-head target recipe passed only as a pre-PPO recipe; fixed-split and walk-forward return/drawdown gates remain untested.'
elif recall >= 0.70:
    decision = 'state_head_target_recipe_recall_ok_false_positive_or_precision_blocked'
    next_mechanism = 'tighten_state_head_threshold_or_calibration_before_ppo'
    blocker = 'State head detects rebound but false positives/precision are not safe enough for a PPO recipe.'
else:
    decision = 'state_head_target_recipe_no_reliable_rebound_recall'
    next_mechanism = 'new_rebound_features_or_fold_local_target_discovery'
    blocker = 'State-head target routing did not recover enough rebound rows.'

selection = {
    'experiment': EXPERIMENT,
    'created_at': base_context['created_at'],
    'source_experiment': 'rebound_representation_separability_v1',
    'direct_comparison_baseline': 'state_objective_state_conditioned_loss_v1',
    'decision': decision,
    'promotion_allowed': False,
    'full_gate': False,
    'recipe_gate': recipe_gate,
    'promotion_blocker': blocker,
    'best_feature_set': str(best.get('feature_set', '')),
    'best_threshold': float(best.get('threshold', np.nan)),
    'best_state_recall': recall,
    'best_state_precision': precision,
    'best_false_positive_rate': fpr,
    'best_target_mae': float(best.get('target_mae', np.nan)),
    'best_rebound_target_mae': float(best.get('rebound_target_mae', np.nan)),
    'best_true_positive': int(best.get('true_positive', 0)),
    'best_false_positive': int(best.get('false_positive', 0)),
    'best_false_negative': int(best.get('false_negative', 0)),
    'best_true_negative': int(best.get('true_negative', 0)),
    'precision_caution': precision < 0.10 if np.isfinite(precision) else True,
    'next_mechanism': next_mechanism,
    'acceptance_gate_for_next_run': {
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'state_head_recipe_gate_required': True,
        'holdout_rebound_accuracy_min': 0.60,
        'no_runtime_promotion_from_recipe_only': True,
        'precision_caution_requires_ppo_drawdown_guard': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/2004.11362',
        'https://arxiv.org/abs/1807.03748',
        'https://arxiv.org/abs/1901.05555',
    ],
    'sidecars': ['base_context.json', 'recipe_scores.csv', 'row_scores.csv', 'summary.json'],
}

summary = {
    'base_context': base_context,
    'best_row': best,
    'selection': selection,
    'conclusion': [
        f"Best recipe {selection['best_feature_set']} threshold={selection['best_threshold']} recovered {selection['best_true_positive']} rebound rows with {selection['best_false_positive']} false positives.",
        f"Recall/precision/FPR were {selection['best_state_recall']:.3f}/{selection['best_state_precision']:.3f}/{selection['best_false_positive_rate']:.3f}.",
        f"Target MAE/rebound MAE were {selection['best_target_mae']:.4f}/{selection['best_rebound_target_mae']:.4f}.",
        f"Decision: {selection['decision']}; next mechanism: {selection['next_mechanism']}.",
        "This is a pre-PPO recipe only; precision is low because rebound rows are extremely sparse, so drawdown/risk gates must be part of the next PPO validation.",
    ],
}

(OUT_DIR / 'base_context.json').write_text(json.dumps(base_context, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n')

print(json.dumps(selection, indent=2, sort_keys=True))


{
  "acceptance_gate_for_next_run": {
    "fixed_split_required": true,
    "holdout_rebound_accuracy_min": 0.6,
    "max_dd_max_pct": 15.0,
    "no_runtime_promotion_from_recipe_only": true,
    "precision_caution_requires_ppo_drawdown_guard": true,
    "relative_return_min_pp": 30.0,
    "state_head_recipe_gate_required": true,
    "walkforward_required": true
  },
  "best_false_negative": 0,
  "best_false_positive": 123,
  "best_false_positive_rate": 0.09808612440191387,
  "best_feature_set": "momentum_drawdown",
  "best_rebound_target_mae": 0.028571428571428574,
  "best_state_precision": 0.05384615384615385,
  "best_state_recall": 1.0,
  "best_target_mae": 0.00030123285216541985,
  "best_threshold": 0.5,
  "best_true_negative": 1131,
  "best_true_positive": 7,
  "created_at": "2026-05-25T13:34:18.440808+00:00",
  "decision": "state_head_target_recipe_passes_pre_ppo_gate_no_runtime_promotion",
  "direct_comparison_baseline": "state_objective_state_conditioned_loss_v1",
  "experiment

## Conclusion

The executed notebook writes the single-run conclusion into `selection.json` and `summary.json`. This card can only hand off a future fold-local PPO recipe with an auxiliary rebound state head; it is not promotion evidence.
